In [1]:
import anndata as ad
import pandas as pd
import numpy as np
import os

# Get FP

In [2]:
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit.DataStructs import ConvertToNumpyArray

def smiles_to_fingerprints(smiles_list, radius=1, fp_size=2000):
    """Convert a list of SMILES strings to Morgan (ECFP) fingerprint matrix.

    Args:
        smiles_list: Iterable of SMILES strings.
        radius: Morgan fingerprint radius (default 1, i.e. ECFP2).
        fp_size: Fingerprint bit vector size.

    Returns:
        np.ndarray of shape (len(smiles_list), fp_size), dtype uint8.
        Invalid SMILES yield a zero vector for that row.
    """
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=fp_size)
    fps = []
    for smile in smiles_list:
        mol = Chem.MolFromSmiles(smile)
        if mol is None:
            fps.append(None)
        else:
            fp = gen.GetFingerprint(mol)
            arr = np.zeros(fp_size, dtype=np.uint8)
            ConvertToNumpyArray(fp, arr)
            fps.append(arr)
    return fps

In [3]:
de_train = ad.read_h5ad('../../data_mol_emb/benchmark/resources/datasets/neurips-2023-data/de_train.h5ad')
de_test = ad.read_h5ad('../../data_mol_emb/benchmark/resources/datasets/neurips-2023-data/de_test.h5ad')

In [4]:
de = ad.concat([de_train, de_test])
sm_smiles = de.obs[['sm_name', 'SMILES']].drop_duplicates()
sm_smiles['ECFP:2'] = smiles_to_fingerprints(sm_smiles['SMILES'])
sm_smiles = sm_smiles.rename(columns = {'SMILES': 'smiles', 'sm_name': 'perturbagen'})

In [5]:
sm_smiles

,perturbagen,smiles,ECFP:2
"NK cells, TIE2 Kinase Inhibitor",TIE2 Kinase Inhibitor,COc1ccc2cc(-c3[nH]c(-c4ccc([S+](C)[O-])cc4)nc3...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
"B cells, MK-5108",MK-5108,O=C(O)[C@]1(Cc2cccc(Nc3nccs3)n2)CC[C@@H](Oc2cc...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
"NK cells, Lapatinib",Lapatinib,CS(=O)(=O)CCNCc1ccc(-c2ccc3ncnc(Nc4ccc(OCc5ccc...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
"B cells, Belinostat",Belinostat,O=C(/C=C/c1cccc(S(=O)(=O)Nc2ccccc2)c1)NO,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
"B cells, Dabrafenib",Dabrafenib,CC(C)(C)c1nc(-c2cccc(NS(=O)(=O)c3c(F)cccc3F)c2...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
...,...,...,...
"B cells, GLPG0634",GLPG0634,O=C(Nc1nc2cccc(-c3ccc(CN4CCS(=O)(=O)CC4)cc3)n2...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
"NK cells, Mubritinib (TAK 165)",Mubritinib (TAK 165),FC(F)(F)c1ccc(/C=C/c2nc(COc3ccc(CCCCn4ccnn4)cc...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
"NK cells, Vanoxerine",Vanoxerine,Fc1ccc(C(OCCN2CCN(CCCc3ccccc3)CC2)c2ccc(F)cc2)cc1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
"NK cells, SB525334",SB525334,Cc1cccc(-c2[nH]c(C(C)(C)C)nc2-c2ccc3nccnc3c2)n1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


# Get Pubchem

In [6]:
op3 = ad.read_h5ad('../../../data/op3/pseudobulk_processed/sep_rep/op3_standardized_processed.h5ad')
op3_obs = op3.obs
df_op3_sm = op3_obs.drop_duplicates(['perturbagen', 'pubchem_cid'])[['perturbagen', 'pubchem_cid']].reset_index(drop=True)

In [7]:
df_op3_sm_add = pd.DataFrame({'perturbagen': ['Belinostat', 'Dabrafenib'],
                              'pubchem_cid': [6918638, 44462760]})

In [8]:
df_emb_op3 = pd.concat([df_op3_sm, df_op3_sm_add]).reset_index(drop=True)

In [9]:
df_emb_op3['pubchem_cid'] = df_emb_op3['pubchem_cid'].astype(str)

In [10]:
df_emb_op3

,perturbagen,pubchem_cid
0,TIE2 Kinase Inhibitor,23625762
1,MK-5108,24748204
2,Lapatinib,208908
3,Dimethyl Sulfoxide,679
4,Atorvastatin,60823
...,...,...
136,Vanoxerine,3455
137,SB525334,9967941
138,HYDROXYUREA,3657
139,Belinostat,6918638


# Get embeddings and JOIN

In [13]:
epoch_dirs = ['epoch_epoch_0019']

In [14]:
epoch_dirs

['epoch_epoch_0019']

In [27]:
import os
os.makedirs('../../data_lpm_style_learning_missed/benchmark/resources/datasets/neurips-2023-data-subsample', exist_ok=True)

for epoch_dir in epoch_dirs:
    df_pert_all = pd.read_pickle(f'../../../lpm_style/files/single_run_all_25_epochs/{epoch_dir}/df_pert.pkl')\
                    .rename(columns={'symbol': 'symbol_all', 
                                     'code': 'code_all', 
                                     'lpm_style_embeddings': 
                                     'lpm_style_embeddings_all'})

    df_pert_l1000 = pd.read_pickle(f'../../../lpm_style/files/single_run_l1000_25_epochs/{epoch_dir}/df_pert.pkl')\
                    .rename(columns={'symbol': 'symbol_l1000', 
                                     'code': 'code_l1000', 
                                     'lpm_style_embeddings': 
                                     'lpm_style_embeddings_l1000'})

    
    df_emb_op3_merged = df_emb_op3.merge(df_pert_all, left_on='pubchem_cid', right_on='symbol_all', how='left')\
                            .merge(df_pert_l1000, left_on='pubchem_cid', right_on='symbol_l1000', how='left')\
                            .merge(sm_smiles, left_on='perturbagen', right_on='perturbagen',  how='left')

    #mask = df_emb_op3_merged['lpm_style_embeddings_all'].isna() | df_emb_op3_merged['ECFP:2'].isna()
    #df_emb_op3_merged_filtered = df_emb_op3_merged[~mask]
    df_emb_op3_merged_filtered = df_emb_op3_merged.copy()
    
    tag = int(epoch_dir.split('_')[-1])
    
    df_emb_op3_all_to_save = df_emb_op3_merged_filtered.reset_index(drop=True)
    df_emb_op3_all_to_save.to_pickle(f'op3_emb_all_l1000_{tag + 1}.pkl')

In [30]:
df_emb_op3_all_to_save.head()

,perturbagen,pubchem_cid,symbol_all,code_all,lpm_style_embeddings_all,symbol_l1000,code_l1000,lpm_style_embeddings_l1000,smiles,ECFP:2
0,TIE2 Kinase Inhibitor,23625762,23625762,8274.0,"[-0.21788546442985535, 0.228135347366333, 0.05...",23625762,1559.0,"[-0.1081283688545227, 0.25940677523612976, -0....",COc1ccc2cc(-c3[nH]c(-c4ccc([S+](C)[O-])cc4)nc3...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,MK-5108,24748204,24748204,8813.0,"[-0.04050781577825546, 0.0016063833609223366, ...",24748204,1867.0,"[0.2892323434352875, 0.09267596900463104, 0.11...",O=C(O)[C@]1(Cc2cccc(Nc3nccs3)n2)CC[C@@H](Oc2cc...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,Lapatinib,208908,208908,7670.0,"[0.02964354120194912, 0.2212171107530594, -0.0...",208908,1342.0,"[-0.1597915142774582, 0.3697046935558319, -0.1...",CS(=O)(=O)CCNCc1ccc(-c2ccc3ncnc(Nc4ccc(OCc5ccc...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,Dimethyl Sulfoxide,679,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Atorvastatin,60823,60823,31249.0,"[0.007660984992980957, 0.05466464161872864, -0...",60823,19377.0,"[0.16961884498596191, 0.33628782629966736, -0....",CC(C)c1c(C(=O)Nc2ccccc2)c(-c2ccccc2)c(-c2ccc(F...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [33]:
pd.read_pickle('../../data_lpm_style_learning_missed_25_epochs/benchmark/resources/datasets/neurips-2023-data-subsample/op3_emb_l1000_20.pkl')

,perturbagen,LPM_emb,smiles,ECFP:2
0,TIE2 Kinase Inhibitor,"[-0.1081283688545227, 0.25940677523612976, -0....",COc1ccc2cc(-c3[nH]c(-c4ccc([S+](C)[O-])cc4)nc3...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,MK-5108,"[0.2892323434352875, 0.09267596900463104, 0.11...",O=C(O)[C@]1(Cc2cccc(Nc3nccs3)n2)CC[C@@H](Oc2cc...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,Lapatinib,"[-0.1597915142774582, 0.3697046935558319, -0.1...",CS(=O)(=O)CCNCc1ccc(-c2ccc3ncnc(Nc4ccc(OCc5ccc...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,Atorvastatin,"[0.16961884498596191, 0.33628782629966736, -0....",CC(C)c1c(C(=O)Nc2ccccc2)c(-c2ccccc2)c(-c2ccc(F...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,Ganetespib (STA-9090),"[0.19747792184352875, 0.010393727570772171, -0...",CC(C)c1cc(-c2n[nH]c(=O)n2-c2ccc3c(ccn3C)c2)c(O...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
...,...,...,...,...
109,Mubritinib (TAK 165),"[0.30055004358291626, 0.24176226556301117, -0....",FC(F)(F)c1ccc(/C=C/c2nc(COc3ccc(CCCCn4ccnn4)cc...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
110,SB525334,"[-0.1603730171918869, -0.40962570905685425, -0...",Cc1cccc(-c2[nH]c(C(C)(C)C)nc2-c2ccc3nccnc3c2)n1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
111,HYDROXYUREA,"[0.15368907153606415, 0.24922992289066315, 0.0...",NC(O)=NO,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
112,Belinostat,"[0.3923093378543854, 0.15980926156044006, -0.0...",O=C(/C=C/c1cccc(S(=O)(=O)Nc2ccccc2)c1)NO,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [28]:
! ls

01_create_embeddings_sci.ipynb
02_create_embeddings_DILI.ipynb
description.ipynb
dili_lpm_style_embeddings_epoch5.pkl
dili_lpm_style_embeddings.pkl
dili_single_run_all_25_epochs
dili_single_run_all_25_epochs.tar.gz
dili_single_run_all_l1000_25_epochs
dili_single_run_all_l1000_25_epochs.tar.gz
dili_single_run_all_l1000_26_50_epochs
dili_single_run_all_l1000_26_50_epochs.tar.gz
embedin_embeddings_fingerprint_lpm_style_all_vs_l1000_test_subsetting.ipynb
embedin_embeddings_fingerprint_lpm_style_all_vs_l1000_train_test_no_subsetting-Copy1.ipynb
embedin_embeddings_fingerprint_lpm_style_all_vs_l1000_train_test_no_subsetting.ipynb
embedin_embeddings_fingerprint_lpm_style_all_vs_l1000_train_test_subsetting.ipynb
embedin_embeddings_fingerprint_lpm_style.ipynb
grep_embeddings_backup.ipynb
grep_embeddings.ipynb
op3_emb_all_20.pkl
op3_emb_all_l1000_20.pkl
op3_emb_all_l100020.pkl
sci_lpm_style_embeddings_epoch5.pkl
sci_lpm_style_embeddings.pkl
sci_single_run_all_25_epochs
sci_single_run_all_25_epoch

In [24]:
df_emb_op3_all_to_save[df_emb_op3_all_to_save['lpm_style_embeddings_all'].isna()]

,perturbagen,pubchem_cid,symbol_all,code_all,lpm_style_embeddings_all,symbol_l1000,code_l1000,lpm_style_embeddings_l1000,smiles,ECFP:2
3,Dimethyl Sulfoxide,679,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
30,BMS-265246,5329775,NaN,NaN,NaN,NaN,NaN,NaN,CCCCOc1c(C(=O)c2c(F)cc(C)cc2F)cnc2[nH]ncc12,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
46,Vardenafil,135400189,NaN,NaN,NaN,NaN,NaN,NaN,CCCc1nc(C)c2c(=O)[nH]c(-c3cc(S(=O)(=O)N4CCN(CC...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [14]:
import os
os.makedirs('../../data_lpm_style_learning_missed/benchmark/resources/datasets/neurips-2023-data-subsample', exist_ok=True)

for epoch_dir in epoch_dirs:
    df_pert_l1000 = pd.read_pickle(f'../../../lpm_style/files/single_run_l1000_26_50_epochs/{epoch_dir}/df_pert.pkl')\
                    .rename(columns={'symbol': 'symbol_l1000', 
                                     'code': 'code_l1000', 
                                     'lpm_style_embeddings': 
                                     'lpm_style_embeddings_l1000'})
    

    
    df_emb_op3_merged = df_emb_op3.merge(df_pert_l1000, left_on='pubchem_cid', right_on='symbol_l1000', how='left')\
                            .merge(sm_smiles, left_on='perturbagen', right_on='perturbagen',  how='left')

    mask = df_emb_op3_merged['lpm_style_embeddings_l1000'].isna() | df_emb_op3_merged['ECFP:2'].isna()
    df_emb_op3_merged_filtered = df_emb_op3_merged[~mask]
    
    tag = int(epoch_dir.split('_')[-1])
    
    df_emb_op3_l1000_to_save = df_emb_op3_merged_filtered.rename(columns={'lpm_style_embeddings_l1000': 'LPM_emb'})[['perturbagen', 'LPM_emb', 'smiles', 'ECFP:2']].reset_index(drop=True)
    df_emb_op3_l1000_to_save.to_pickle(f'../../data_lpm_style_learning_missed/benchmark/resources/datasets/neurips-2023-data-subsample/op3_emb_l1000_{tag + 1}.pkl')

In [24]:
df_pert_all = pd.read_pickle(f'../../../lpm_style/files/single_run/epoch_epoch_0024/df_pert.pkl')\
                    .rename(columns={'symbol': 'symbol_all', 
                                     'code': 'code_all', 
                                     'lpm_style_embeddings': 
                                     'lpm_style_embeddings_all'})

    
df_emb_op3_merged = df_emb_op3.merge(df_pert_all, left_on='pubchem_cid', right_on='symbol_all', how='left')\
                        .merge(sm_smiles, left_on='perturbagen', right_on='perturbagen',  how='left')

mask = df_emb_op3_merged['ECFP:2'].isna()
df_emb_op3_merged_filtered = df_emb_op3_merged[~mask]

tag = int(epoch_dir.split('_')[-1])

df_emb_op3_all_to_save = df_emb_op3_merged_filtered.rename(columns={'lpm_style_embeddings_all': 'LPM_emb'})[['perturbagen', 'LPM_emb', 'smiles', 'ECFP:2']].reset_index(drop=True)
df_emb_op3_all_to_save.to_pickle(f'../../data_lpm_style_learning_missed/benchmark/resources/datasets/neurips-2023-data-subsample/op3_emb_fp.pkl')

# SPLIT

In [ ]:
import anndata as ad
import pandas as pd
import pickle

In [2]:
import pandas as pd


In [3]:
df = pd.read_csv('../../data_lpm_style_learning_missed_25_epochs/benchmark/resources/datasets/neurips-2023-data-subsample/id_map.csv')

In [6]:
len(df['sm_name'].unique())

35

In [7]:
35/140

0.25

In [26]:
import numpy as np

In [27]:
ratio = 0.25

In [28]:
op3_train_subsample = ad.read_h5ad('../../data_mol_emb/benchmark/resources/datasets/neurips-2023-data/de_train.h5ad')
op3_test_subsample = ad.read_h5ad('../../data_mol_emb/benchmark/resources/datasets/neurips-2023-data/de_test.h5ad')
df = pd.read_csv('../../data_mol_emb/benchmark/resources/datasets/neurips-2023-data/id_map.csv')

In [29]:
#path = '../../data_lpm_style_learning_missed/benchmark/resources/datasets/neurips-2023-data-subsample/op3_emb_all_5.pkl'
#with open(path, 'rb') as fp:
#    op3_emb = pickle.load(fp)

In [53]:
op3_train_subsample = op3_train_subsample[op3_train_subsample.obs['sm_name'].isin(op3_emb['perturbagen'])].copy()
op3_test_subsample = op3_test_subsample[op3_test_subsample.obs['sm_name'].isin(op3_emb['perturbagen'])].copy()

In [30]:
op3_test_subsample

AnnData object with n_obs × n_vars = 151 × 5317
    obs: 'sm_cell_type', 'cell_type', 'sm_name', 'sm_lincs_id', 'SMILES', 'split', 'control'
    uns: 'dataset_description', 'dataset_id', 'dataset_name', 'dataset_organism', 'dataset_reference', 'dataset_summary', 'dataset_url', 'single_cell_obs'
    layers: 'AveExpr', 'B', 'P.Value', 'adj.P.Value', 'clipped_sign_log10_pval', 'is_de', 'is_de_adj', 'logFC', 'sign_log10_adj_pval', 'sign_log10_pval', 't'

In [31]:
op3_subsample = ad.concat([op3_train_subsample, op3_test_subsample], uns_merge='same')

In [32]:
df_single_cell_obs = pd.concat([op3_train_subsample.uns['single_cell_obs'], op3_test_subsample.uns['single_cell_obs']])

In [36]:
compounds = np.array(op3_subsample.obs['sm_name'].unique())

In [37]:
np.random.seed(42)
test_sample = np.random.choice(compounds, size=int(len(compounds) * ratio), replace=False)

In [38]:
op3_subsample.obs['new_split'] = np.where(op3_subsample.obs['sm_name'].isin(test_sample), 'test', 'train')

In [39]:
op3_train_subsample_ = op3_subsample[op3_subsample.obs['new_split'] == 'train'].copy()
op3_train_subsample_.uns['single_cell_obs'] = df_single_cell_obs[~df_single_cell_obs['sm_name'].isin(test_sample)]
op3_train_subsample_.write_h5ad('../../data_lpm_style_learning_missed/benchmark/resources/datasets/neurips-2023-data-subsample/de_train.h5ad', compression='gzip')

In [40]:
op3_test_subsample_ = op3_subsample[op3_subsample.obs['new_split'] == 'test'].copy()
op3_test_subsample_.uns['single_cell_obs'] = df_single_cell_obs[df_single_cell_obs['sm_name'].isin(test_sample)]
op3_test_subsample_.write_h5ad('../../data_lpm_style_learning_missed/benchmark/resources/datasets/neurips-2023-data-subsample/de_test.h5ad', compression='gzip')

In [41]:
op3_test_subsample_.obs[['sm_name', 'cell_type']].reset_index(drop=True).reset_index().rename(columns={'index': 'id'}).to_csv('../../data_lpm_style_learning_missed/benchmark/resources/datasets/neurips-2023-data-subsample/id_map.csv', index=False)

In [45]:
op3_test_subsample_[op3_test_subsample_.obs['sm_name'].isin(op3_train_subsample_.obs['sm_name'])]

View of AnnData object with n_obs × n_vars = 0 × 5317
    obs: 'sm_cell_type', 'cell_type', 'sm_name', 'sm_lincs_id', 'SMILES', 'split', 'control', 'new_split'
    uns: 'dataset_description', 'dataset_id', 'dataset_name', 'dataset_organism', 'dataset_reference', 'dataset_summary', 'dataset_url', 'single_cell_obs'
    layers: 'AveExpr', 'B', 'P.Value', 'adj.P.Value', 'clipped_sign_log10_pval', 'is_de', 'is_de_adj', 'logFC', 'sign_log10_adj_pval', 'sign_log10_pval', 't'

In [47]:
len(op3_test_subsample_.obs['sm_name'].unique()) + len(op3_train_subsample_.obs['sm_name'].unique())

140

# Check emb

In [63]:
op3_train_subsample_

AnnData object with n_obs × n_vars = 341 × 5317
    obs: 'sm_cell_type', 'cell_type', 'sm_name', 'sm_lincs_id', 'SMILES', 'split', 'control', 'new_split'
    uns: 'dataset_description', 'dataset_id', 'dataset_name', 'dataset_organism', 'dataset_reference', 'dataset_summary', 'dataset_url', 'single_cell_obs'
    layers: 'AveExpr', 'B', 'P.Value', 'adj.P.Value', 'clipped_sign_log10_pval', 'is_de', 'is_de_adj', 'logFC', 'sign_log10_adj_pval', 'sign_log10_pval', 't'

In [64]:
adata_train_prev = ad.read_h5ad('../../data_lpm_stype_epoch1/benchmark/resources/datasets/neurips-2023-data-subsample/de_train.h5ad')

In [65]:
adata_test_prev = ad.read_h5ad('../../data_lpm_stype_epoch1/benchmark/resources/datasets/neurips-2023-data-subsample/de_test.h5ad')

In [66]:
op3_pickle = pd.read_pickle('../../data_lpm_stype_epoch1/benchmark/resources/datasets/neurips-2023-data-subsample/op3_emb.pkl')

In [67]:
(op3_test_subsample_.layers['clipped_sign_log10_pval'] == adata_test_prev.layers['clipped_sign_log10_pval']).all()

ValueError: operands could not be broadcast together with shapes (109,5317) (134,5317) 

In [75]:
(op3_train_subsample_.layers['clipped_sign_log10_pval'] == adata_train_prev.layers['clipped_sign_log10_pval']).all()

True

In [82]:
(op3_emb[['ECFP:2']].astype(str) == op3_pickle[['ECFP:2']].astype(str)).all()

ECFP:2    True
dtype: bool

In [74]:
op3_emb[['perturbagen', 'smiles', ]].compare(op3_pickle[['perturbagen', 'smiles']])

Empty DataFrame
Columns: []
Index: []